In [1]:
##DSQN plays second in this code and this code is good to use. 
##During evaluation didnt give good results. this the code above only. so testing with increased 
##replay buffer memory size. 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from IPython.display import display
from itertools import count
import wandb

# Initialize WandB
wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="connect4 dsqn going second")

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Connect Four Environment
class ConnectX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1, 'draw': 0.5, 'lose': -1, 'step': -0.05}

    def render(self):
        rendered_board = self.board_state.astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = 'O'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id):
                    return True
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id):
                    return True
        for i in range(self.board_height - 3):
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]):
                    return True
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]):
                    return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['draw']
        return self.reward['step']

    def make_move(self, action, player):
        if action not in self.get_available_actions():
            print('Move is invalid')
            return self.board_state.copy(), self.reward['lose'], False
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

    def check_potential_win(self, action, player):
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        if i < 0:
            return False
        original_value = self.board_state[i, action]
        self.board_state[i, action] = self.players[player]
        is_win = self.check_win(player)
        self.board_state[i, action] = original_value
        return is_win

# Replay Memory
class ReplayMemory:
    def __init__(self, capacity=100000):
        self.memory = []
        self.capacity = capacity
        self.position = 0

    def dump(self, transition):
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

# Surrogate Gradient Spike Function
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# Deep Spiking Neural Network (DSNN)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta

        torch.manual_seed(seed)

        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))

        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        mem = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        spk = [[] for _ in range(len(self.weights))]
        spike_counts = [0] * (len(self.weights) - 1)

        mem_rec = []
        for t in range(self.simulation_time):
            input_t = x[:, t, :]
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.matmul(input_t, self.weights[l])
                else:
                    h = torch.matmul(spk[l-1][-1], self.weights[l])

                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]

                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                    spike_counts[l] += spk_current.sum().item()

                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])

        q_values = mem_rec[-1]
        return q_values, mem_rec, spk

# DSQN Agent
class DSQN:
    def __init__(self, discount_factor, dsnn_config):
        self.gamma = discount_factor
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.update_target_network()
        
        self.population_size = 3
        self.board_size = 6 * 7
        self.neurons_per_position = {
            0: [0.2, 0.8, 0.2],
            1: [0.9, 0.2, 0.1],
            2: [0.1, 0.2, 0.9]
        }

    def population_encode(self, states):
        batch_size = states.size(0)
        encoded = torch.zeros(batch_size, self.simulation_time, self.board_size * self.population_size, device=device)
        
        states_flat = states.view(batch_size, -1).long()
        pop_code = torch.zeros(batch_size, self.board_size * self.population_size, device=device)
        
        for pos in range(self.board_size):
            for val in range(3):
                mask = (states_flat[:, pos] == val)
                for neuron in range(self.population_size):
                    pop_code[mask, pos * self.population_size + neuron] = self.neurons_per_position[val][neuron]
        
        noise = torch.normal(mean=0.0, std=0.05, size=(batch_size, self.simulation_time, self.board_size * self.population_size), device=device)
        for t in range(self.simulation_time):
            encoded[:, t, :] = pop_code + noise[:, t, :]
        
        return encoded

    def select_action(self, state, available_actions, training=True, steps_done=None):
        state = torch.tensor(state, dtype=torch.float, device=device).unsqueeze(0)
        encoded_state = self.population_encode(state)
        
        if training:
            if steps_done is None:
                raise ValueError("steps_done is required when training=True")
            eps_threshold = EPS_END + (EPS_START - EPS_END) * np.exp(-steps_done / EPS_DECAY)
        else:
            eps_threshold = 0
        
        if random.random() > eps_threshold:
            with torch.no_grad():
                q_values, _, _ = self.training_net(encoded_state)
                q_values = q_values.flatten()
                valid_q = q_values[available_actions]
                return available_actions[torch.argmax(valid_q).item()]
        return random.choice(available_actions)

    def optimize_model(self, memory):
        if len(memory) < self.batch_size:
            return
        
        transitions = memory.sample(self.batch_size)
        state_batch, action_batch, reward_batch, next_state_batch = zip(*transitions)
        
        state_batch = torch.tensor(np.array(state_batch), dtype=torch.float, device=device)
        action_batch = torch.tensor(action_batch, dtype=torch.long, device=device)
        reward_batch = torch.tensor(reward_batch, dtype=torch.float, device=device)
        non_final_mask = torch.tensor([s is not None for s in next_state_batch], device=device)
        non_final_next_states = [s for s in next_state_batch if s is not None]
        non_final_next_states = torch.tensor(np.array(non_final_next_states), dtype=torch.float, device=device) if non_final_next_states else None
        
        state_batch_encoded = self.population_encode(state_batch)
        current_q, _, _ = self.training_net(state_batch_encoded)
        current_q = current_q.gather(1, action_batch.unsqueeze(1)).squeeze(1)
        
        next_state_values = torch.zeros(self.batch_size, device=device)
        if non_final_next_states is not None:
            non_final_next_states_encoded = self.population_encode(non_final_next_states)
            with torch.no_grad():
                next_q_values, _, _ = self.target_net(non_final_next_states_encoded)
                next_state_values[non_final_mask] = next_q_values.max(1)[0]
        
        expected_q = reward_batch + (self.gamma * next_state_values)
        loss = F.smooth_l1_loss(current_q, expected_q)
        
        self.training_net.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.training_net.parameters(), max_norm=1.0)
        self.training_net.optimizer.step()

    def update_target_network(self):
        self.target_net.load_state_dict(self.training_net.state_dict())

# Random Agent
def random_agent(actions):
    return random.choice(actions)

# Evaluation Function
def evaluate_policy(agent, env, episodes=100):
    wins, losses, draws, moves_taken = 0, 0, 0, []
    for _ in range(episodes):
        state = env.reset()
        move_count = 0
        while not env.isDone:
            # Random agent plays as 'X' (p2)
            action = random_agent(env.get_available_actions())
            state, reward, _ = env.make_move(action, 'p2')
            move_count += 1
            if env.isDone:
                if reward == 1:  # 'X' wins
                    losses += 1
                elif reward == 0.5:  # Draw
                    draws += 1
                break
            # DSQN agent plays as 'O' (p1)
            action = agent.select_action(state, env.get_available_actions(), training=False)
            state, reward, valid = env.make_move(action, 'p1')
            move_count += 1
            if not valid:
                losses += 1
                break
            if env.isDone:
                if reward == 1:  # 'O' wins
                    wins += 1
                    moves_taken.append(move_count)
                elif reward == 0.5:  # Draw
                    draws += 1
                break
    return wins / episodes, losses / episodes, draws / episodes, np.mean(moves_taken) if moves_taken else 0

# Hyperparameters
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TARGET_UPDATE = 10
MEMORY_CAPACITY = 100000
NUM_EPISODES = 30000
LOG_INTERVAL = 100

# DSNN configuration
dsnn_config = {
    'architecture': [6 * 7 * 3, 128, 128, 7],
    'seed': 42,
    'alpha': 0.9,
    'beta': 0.8,
    'weight_scale': 1.0,
    'batch_size': BATCH_SIZE,
    'threshold': 0.1,
    'simulation_time': 1,
    'learning_rate': 0.0005,
    'reset_potential': 0.0
}

# Initialize environment and agent
env = ConnectX()
agent = DSQN(discount_factor=GAMMA, dsnn_config=dsnn_config)
memory = ReplayMemory()

def print_spikes(last_spikes):
    """Print spike activity for the last timestep per neuron for each layer"""
    print("\nSpike Activity:")
    for layer_idx, layer_spikes in enumerate(last_spikes):
        if layer_spikes is not None and layer_spikes.size(0) > 0:
            spike_pattern = layer_spikes[0].cpu().numpy()
            print(f"Layer {layer_idx + 1}, Spikes: {spike_pattern}")
        else:
            print(f"Layer {layer_idx + 1}, Spikes: []")

def print_encoded_state(encoded_state):
    """Print population encoding for the Connect Four board (6x7 grid) at the first timestep"""
    print("\nPopulation Encoding:")
    encoded_np = encoded_state[0, 0, :].cpu().numpy()
    for row in range(6):
        print()
        for col in range(7):
            pos = row * 7 + col
            start_idx = pos * 3
            neurons = encoded_np[start_idx:start_idx + 3]
            print(f"[{neurons[0]:.1f} {neurons[1]:.1f} {neurons[2]:.1f}]", end=" ")
    print()

def print_membrane_potentials(q_values, action):
    """Show output layer membrane potentials (Q-values) and selected action"""
    print("\nOutput Q-Values (final timestep, all actions):")
    q_np = q_values.cpu().numpy().flatten()
    for i in range(7):
        print(f"Action {i}: {q_np[i]:.6f}")
    print(f"Selected Action: {action} (Q-value: {q_np[action]:.6f})")

# Training loop
steps_done = 0
training_history = []
best_win_rate = 0
interval_wins = 0
interval_losses = 0
interval_draws = 0
interval_total_moves = 0

for episode in range(NUM_EPISODES):
    state = env.reset()
    move_count = 0
    outcome = None
    last_dqn_state = None
    last_dqn_action = None

    for t in count():
        # Random agent plays as 'X' (p2) first
        available_actions = env.get_available_actions()
        action_p2 = random_agent(available_actions)
        next_state, reward_p2, _ = env.make_move(action_p2, 'p2')
        move_count += 1
        # if episode % 1 == 0:
        #     print(f"Episode {episode}, Move {t+1}, Player 'X' Action: {action_p2}")
        #     env.render()
        if env.isDone:
            if reward_p2 == 1:  # 'X' wins
                outcome = 'loss'
                reward = -1
                # if episode % 1 == 0:
                #     print(f"Episode {episode} ended: X wins!")
                #     env.render()
            elif reward_p2 == 0.5:  # Draw
                outcome = 'draw'
                reward = 0.5
                # if episode % 1 == 0:
                #     print(f"Episode {episode} ended: Draw!")
                #     env.render()
            if last_dqn_state is not None:  # Store transition if DSQN made a move
                memory.dump((last_dqn_state, last_dqn_action, reward, None))
            break
        # # DSQN agent plays as 'O' (p1)
        # if episode % 5 == 0:
        #     state_tensor = torch.tensor(next_state, dtype=torch.float, device=device).unsqueeze(0)
        #     encoded_state = agent.population_encode(state_tensor)
        #     with torch.no_grad():
        #         q_values, _, spk = agent.training_net(encoded_state)
        #     last_spikes = [spk[l][-1] if spk[l] else torch.zeros(1, agent.training_net.weights[l].size(1), device=device) for l in range(len(agent.training_net.weights)-1)]
        #     action = agent.select_action(next_state, env.get_available_actions(), training=True, steps_done=steps_done)
        #     print_spikes(last_spikes)
        #     print_encoded_state(encoded_state)
        #     print_membrane_potentials(q_values, action)
        available_actions = env.get_available_actions()
        action = agent.select_action(next_state, available_actions, training=True, steps_done=steps_done)
        steps_done += 1
        next_next_state, reward, valid = env.make_move(action, 'p1')
        move_count += 1
        # if episode % 1 == 0:
        #     print(f"Episode {episode}, Move {t+1}, Player 'O' Action: {action}")
        #     env.render()
        if not valid:
            outcome = 'loss'
            memory.dump((next_state, action, reward, None))
            # if episode % 1 == 0:
            #     print(f"Episode {episode} ended: Invalid move by 'O'!")
            #     env.render()
            break
        if env.isDone:
            if reward == 1:  # 'O' wins
                outcome = 'win'
                # if episode % 1 == 0:
                #     print(f"Episode {episode} ended: O wins!")
                #     env.render()
            elif reward == 0.5:  # Draw
                outcome = 'draw'
                # if episode % 1 == 0:
                #     print(f"Episode {episode} ended: Draw!")
                #     env.render()
            memory.dump((next_state, action, reward, None))
            break
        memory.dump((next_state, action, reward, next_next_state))
        last_dqn_state = next_state
        last_dqn_action = action
        state = next_next_state
        agent.optimize_model(memory)

    if outcome == 'win':
        interval_wins += 1
    elif outcome == 'loss':
        interval_losses += 1
    elif outcome == 'draw':
        interval_draws += 1
    interval_total_moves += move_count

    if episode % TARGET_UPDATE == 0:
        agent.update_target_network()

    if (episode + 1) % LOG_INTERVAL == 0 or episode == NUM_EPISODES - 1:
        total_games_in_interval = episode % LOG_INTERVAL + 1 if episode == NUM_EPISODES - 1 and (episode + 1) % LOG_INTERVAL != 0 else LOG_INTERVAL
        if total_games_in_interval > 0:
            interval_win_rate = interval_wins / total_games_in_interval
            interval_loss_rate = interval_losses / total_games_in_interval
            interval_draw_rate = interval_draws / total_games_in_interval
            interval_win_draw_rate = interval_win_rate + interval_draw_rate
            interval_avg_moves = interval_total_moves / total_games_in_interval

            print(f"\n--- Episode {episode + 1}/{NUM_EPISODES} ---")
            print(f"Training Summary (last {total_games_in_interval} episodes, 'O' perspective):")
            print(f" 'O' Wins: {interval_wins}, Losses: {interval_losses}, Draws: {interval_draws}")
            print(f" Win Rate: {interval_win_rate:.3f}, Loss Rate: {interval_loss_rate:.3f}, Draw Rate: {interval_draw_rate:.3f}, Win+Draw Rate: {interval_win_draw_rate:.3f}")
            print(f" Avg Moves per game: {interval_avg_moves:.1f}")

            wandb.log({
                "Episode": episode + 1,
                "Wins": interval_wins,
                "Losses": interval_losses,
                "Draws": interval_draws,
                "Win Rate": interval_win_rate,
                "Loss Rate": interval_loss_rate,
                "Draw Rate": interval_draw_rate,
                "Win+Draw Rate": interval_win_draw_rate,
                "Training/Interval Avg Moves": interval_avg_moves,
            })

        interval_wins = 0
        interval_losses = 0
        interval_draws = 0
        interval_total_moves = 0
        
wandb.finish()
print("Training complete")
# torch.save({
#     'training_net': agent.training_net.state_dict(),
#     'target_net': agent.target_net.state_dict()
# }, 'DSQN_connect4_O_second_increplaybuffer.pth')
# print("Trained model saved to 'DSQN_connect4_O_second_increplaybuffer.pth'")
model_save_path = "../saved_models/DSQN_connect4_second.pth"
# Save final model
torch.save({
    'training_net': agent.training_net.state_dict(),
    'target_net': agent.target_net.state_dict()
}, model_save_path)
print("Trained model saved to '../saved_models/DSQN_connect4_second.pth'")

# Demo function
def demo():
    env.reset()
    env.render()
    while not env.isDone:
        # Random agent plays as 'X' (p2)
        state = env.board_state.copy()
        action = random_agent(env.get_available_actions())
        state, reward, _ = env.make_move(action, 'p2')
        env.render()
        if reward == 1:
            print("X Wins!")
            break
        if reward == 0.5:
            print("Draw!")
            break
        # DSQN agent plays as 'O' (p1)
        action = agent.select_action(state, env.get_available_actions(), training=False)
        state, reward, valid = env.make_move(action, 'p1')
        env.render()
        if not valid:
            print("Invalid move by 'O'!")
            break
        if reward == 1:
            print("O Wins!")
            break
        if reward == 0.5:
            print("Draw!")
            break

demo()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/saideepa0501/.netrc
wandb: Currently logged in as: kradeero (kradeero-ohio-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cpu

--- Episode 100/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 45, Losses: 54, Draws: 1
 Win Rate: 0.450, Loss Rate: 0.540, Draw Rate: 0.010, Win+Draw Rate: 0.460
 Avg Moves per game: 20.2

--- Episode 200/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 44, Losses: 56, Draws: 0
 Win Rate: 0.440, Loss Rate: 0.560, Draw Rate: 0.000, Win+Draw Rate: 0.440
 Avg Moves per game: 18.3

--- Episode 300/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 49, Losses: 51, Draws: 0
 Win Rate: 0.490, Loss Rate: 0.510, Draw Rate: 0.000, Win+Draw Rate: 0.490
 Avg Moves per game: 19.1

--- Episode 400/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 66, Losses: 34, Draws: 0
 Win Rate: 0.660, Loss Rate: 0.340, Draw Rate: 0.000, Win+Draw Rate: 0.660
 Avg Moves per game: 15.9

--- Episode 500/30000 ---
Training Summary (last 100 episodes, 'O' perspective):
 'O' Wins: 66, Losses: 3

Draw Rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Draws,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Episode,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
Loss Rate,█▇▄▅▃▄▃▄▂▂▃▂▂▃▂▂▃▂▃▃▁▃▃▃▂▂▂▂▂▂▂▂▂▃▂▂▂▁▂▁
Losses,█▅▇▅▆▅▇▅▇▅▃▄▄▆▄▄▄▄▄▅▅▃▆▂▃▁▅▃▅▂▂▃▃▂▄▃▂▃▅▅
Training/Interval Avg Moves,█▆▆▅▆▄▄▅▅▃▅▃▃▅▅▄▅▆▅▅▂▅▁▃▄▅▅▅▃▄▃▅▃▄▃▃▄▄▂▃
Win Rate,▁▂▃▅▆▆▅▆▅▆▆▇▆▆▆▆▇▆▆▅▆▇▆▆▆▆▆▆▇▆▆▆▆█▆█▇▇▆▇
Win+Draw Rate,▁▅▇▆▇▆▆▅▆▇▆▇▇▇▇▇▇▇▆▇▇█▇▇█▇▆████▇▇▇▇▇▆▇██
Wins,▁▂▃▃▃▂▄▃▃▃▄▆▅▅▄▅▄▆▃▅▅▇▄▄▆▄▆█▆▅▂▄▄▇▅▇▂▃▅█
Draw Rate,0
Draws,0


Training complete
Trained model saved to '../saved_models/DSQN_connect4_second.pth'


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,,,,
4,,,,,,,
5,,,,,,,


...

,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,,,,
4,,,,O,,,
5,X,,X,O,,,X


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,O,,,
4,,,,O,,,
5,X,,X,O,,,X


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,,,,
3,,,,O,,,
4,,,,O,,,X
5,X,,X,O,,,X


,0,1,2,3,4,5,6
0,,,,,,,
1,,,,,,,
2,,,,O,,,
3,,,,O,,,
4,,,,O,,,X
5,X,,X,O,,,X


O Wins!
